# **1. Setup**



In [1]:
!pip install datasets scikit-learn torch -q

import torch
from torch.utils.data import Dataset, DataLoader
import numpy as np

from datasets import load_dataset
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import precision_recall_fscore_support, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

torch.manual_seed(42)
np.random.seed(42)

Using device: cpu


# **2. Data Acquisition**



In [2]:
from datasets import load_dataset
from sklearn.model_selection import train_test_split

# Load datasets
train_dataset = load_dataset("UTAustin-AIHealth/MedHallu", "pqa_artificial")['train']
labeled_dataset = load_dataset("UTAustin-AIHealth/MedHallu", "pqa_labeled")['train']

# Subsample labeled dataset for val/test split. 50% validation 50% test. 1000 rows -> 500 val, 500 test
labeled_list = [dict(x) for x in labeled_dataset]
val_orig, test_orig = train_test_split(labeled_list, test_size=0.5, random_state=42)

# Convert artificial dataset to list for preprocessing
train_list = [dict(x) for x in train_dataset]

print(f"Train size: {len(train_list)}, Val size: {len(val_orig)}, Test size: {len(test_orig)}")

Train size: 9000, Val size: 500, Test size: 500


In [3]:
# Look at the first few examples
for i in range(4):
  print("Question: ", train_dataset['Question'][i])
  print("Knowledge: ", train_dataset['Knowledge'][i])
  print("Ground Truth: ", train_dataset['Ground Truth'][i])
  print("Hallucinated Answer: ", train_dataset['Hallucinated Answer'][i])
  print("Category of Hallucination: ", train_dataset['Category of Hallucination'][i])
  print()

Question:  Are group 2 innate lymphoid cells ( ILC2s ) increased in chronic rhinosinusitis with nasal polyps or eosinophilia?
Knowledge:  ['Chronic rhinosinusitis (CRS) is a heterogeneous disease with an uncertain pathogenesis. Group 2 innate lymphoid cells (ILC2s) represent a recently discovered cell population which has been implicated in driving Th2 inflammation in CRS; however, their relationship with clinical disease characteristics has yet to be investigated.', 'The aim of this study was to identify ILC2s in sinus mucosa in patients with CRS and controls and compare ILC2s across characteristics of disease.', 'A cross-sectional study of patients with CRS undergoing endoscopic sinus surgery was conducted. Sinus mucosal biopsies were obtained during surgery and control tissue from patients undergoing pituitary tumour resection through transphenoidal approach. ILC2s were identified as CD45(+) Lin(-) CD127(+) CD4(-) CD8(-) CRTH2(CD294)(+) CD161(+) cells in single cell suspensions thro

# **3. Preprocess dataset**

In [4]:
def preprocess(example):
    processed = []
    question = example["Question"]
    knowledge = " ".join(example["Knowledge"]) if isinstance(example["Knowledge"], list) else example["Knowledge"]
    gt_answer = example["Ground Truth"]
    hallucinated_answer = example["Hallucinated Answer"]

    if not gt_answer or not hallucinated_answer:
        return []

    premise = f"Question: {question} Context: {knowledge}"

    # Positive example (entailed)
    processed.append({"text": f"{premise} [SEP] {gt_answer}", "label": 0})
    # Negative example (hallucinated)
    processed.append({"text": f"{premise} [SEP] {hallucinated_answer}", "label": 1})

    return processed

def build_dataset(split):
    processed = []
    for s in split:
        processed.extend(preprocess(s)) # [ {text, label}, {text, label}, ... ]
    return processed


processed_train = build_dataset(train_list)
processed_val   = build_dataset(val_orig)
processed_test  = build_dataset(test_orig)

# **4. Convert text to vectors (TF-IDF) (Term Frequency-Inverse Document Frequency)**
how often word appears in this text * how rare the word is overall

In [5]:
# convert text to numbers, keeps top 2000 important words. Vocabulary size = 2000
vectorizer = TfidfVectorizer(max_features=2000)

train_texts = [x["text"] for x in processed_train]
val_texts   = [x["text"] for x in processed_val]
test_texts  = [x["text"] for x in processed_test]

# learn vocabulary (fit - The vectorizer scans all training text and builds a dictionary of words.) 
# and transforms text (TF-IDF(blood)...) into numbers. where each number is importance of a word
# fit_transform ONLY on training data
X_train = vectorizer.fit_transform(train_texts) # {(3, 0.8), (10, 0.5)} - Each row = one document. Each pair = (word_index, importance).
# apply same vocabulary to val and test
# transform on validation/test
X_val   = vectorizer.transform(val_texts)
X_test  = vectorizer.transform(test_texts)

# normalizes features so model trains better. w/o scaling some features dominate, with scaling balanced learning.
# (value - mean) / std
scaler = StandardScaler(with_mean=False) # Do NOT subtract the mean only divide by standard deviation. Because your TF-IDF matrix is sparse (mostly zeros) and all zeros will become non-zero values and dense increasing memory.
X_train = scaler.fit_transform(X_train) 
X_val   = scaler.transform(X_val)
X_test  = scaler.transform(X_test)

# covert sparse matrix to dense array
X_train = X_train.toarray() # [0, 0, 0, 0.8, 0, 0, 0, 0, 0, 0, 0.5, 0, ...] full vector length 2000
X_val   = X_val.toarray()
X_test  = X_test.toarray()

y_train = np.array([x["label"] for x in processed_train])
y_val   = np.array([x["label"] for x in processed_val])
y_test  = np.array([x["label"] for x in processed_test])

# **5. Dataset class**

In [6]:
class SimpleDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32) # [0.1, 0.3, ..., 0]
        self.y = torch.tensor(y, dtype=torch.long) # 1

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return {
            "features": self.X[idx],
            "labels": self.y[idx]
        }


train_dataset = SimpleDataset(X_train, y_train)
val_dataset   = SimpleDataset(X_val, y_val)
test_dataset  = SimpleDataset(X_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=8)
test_loader  = DataLoader(test_dataset, batch_size=8)


# **6. Model and optimizer**

In [7]:
import torch.nn as nn

class SimpleClassifier(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(input_dim, 256), # Each of the 256 neurons learns a combination of the 2000 TF-IDF features.
            nn.ReLU(), # add non-linearity. Apply ReLU (max(0, x)) sets negative values to 0, keeps positives
            nn.Dropout(0.2), # randomly turn off 20% of neurons during training to prevent overfitting
            nn.Linear(256, 128), #  compresses the 256 hidden concepts into 128 more abstract concepts
            nn.ReLU(), # add more non-linearity
            nn.Linear(128, 2) # class scores. index 0 -> Correct, index 1 -> Hallucinated
        )

    def forward(self, x):
        return self.model(x)

model = SimpleClassifier(input_dim=2000).to(device)

# optizmer and loss fn
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(y_train),
    y=y_train
)

class_weights = torch.tensor(class_weights, dtype=torch.float32).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=5e-4) # 0.0005
loss_fn = nn.CrossEntropyLoss(weight=class_weights)

In [8]:
for name, param in model.named_parameters():
    print(name)

model.0.weight
model.0.bias
model.3.weight
model.3.bias
model.5.weight
model.5.bias


# **7. Training function**



In [9]:
def train(loader):
    model.train()
    total_loss = 0
    all_preds = []
    all_labels = []

    for batch in loader:
        optimizer.zero_grad()

        inputs = batch["features"].to(device) # shape: [8, 2000]
        labels = batch["labels"].to(device) # shape: [8]

        logits = model(inputs) # example: [2.3, -1.2] 2 class scores per sample . shape: [8, 2000]
        loss = loss_fn(logits, labels) # scalar

        loss.backward() # Compute gradients of loss w.r.t. all model parameters
        optimizer.step() # Uses gradients to update each parameter

        total_loss += loss.item()

        preds = torch.argmax(logits, dim=1)

        all_preds.extend(preds.cpu().numpy()) # example: [0,1,1,0,1,0,1,0]
        all_labels.extend(labels.cpu().numpy()) # example: [0,1,0,0,1,0,1,0]

    acc = np.mean(np.array(all_preds) == np.array(all_labels)) * 100
    avg_loss = total_loss / len(loader)

    return acc, avg_loss

# **8. Evaluation function**

In [10]:
def evaluate(loader):
    model.eval()

    total_loss = 0
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for batch in loader:
            inputs = batch["features"].to(device)
            labels = batch["labels"].to(device)

            logits = model(inputs)
            loss = loss_fn(logits, labels)

            total_loss += loss.item()

            preds = torch.argmax(logits, dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    acc = np.mean(np.array(all_preds) == np.array(all_labels)) * 100
    avg_loss = total_loss / len(loader)

    precision, recall, f1, _ = precision_recall_fscore_support(
        all_labels, all_preds, average='binary', zero_division=0
    )

    return acc, precision, recall, f1, avg_loss

# **9. Training Loop**

In [11]:
epochs = 5

for epoch in range(1, epochs + 1):

    train_acc, train_loss = train(train_loader)
    val_acc, val_prec, val_rec, val_f1, val_loss = evaluate(val_loader)

    print(f"\nEpoch {epoch}")
    print(f"  Train Acc: {train_acc:.2f}%")
    print(f"  Train Loss: {train_loss:.4f}")
    print(f"  Val Acc: {val_acc:.2f}%")
    print(f"  Val Precision: {val_prec:.4f}")
    print(f"  Val Recall:    {val_rec:.4f}")
    print(f"  Val F1 Score:  {val_f1:.4f}")
    print(f"  Val Loss: {val_loss:.4f}")


Epoch 1
  Train Acc: 65.12%
  Train Loss: 0.6121
  Val Acc: 70.80%
  Val Precision: 0.7131
  Val Recall:    0.6960
  Val F1 Score:  0.7045
  Val Loss: 0.5717

Epoch 2
  Train Acc: 72.77%
  Train Loss: 0.5316
  Val Acc: 71.40%
  Val Precision: 0.6945
  Val Recall:    0.7640
  Val F1 Score:  0.7276
  Val Loss: 0.5624

Epoch 3
  Train Acc: 74.70%
  Train Loss: 0.4928
  Val Acc: 72.00%
  Val Precision: 0.7750
  Val Recall:    0.6200
  Val F1 Score:  0.6889
  Val Loss: 0.5827

Epoch 4
  Train Acc: 76.34%
  Train Loss: 0.4468
  Val Acc: 71.30%
  Val Precision: 0.7223
  Val Recall:    0.6920
  Val F1 Score:  0.7068
  Val Loss: 0.6263

Epoch 5
  Train Acc: 78.58%
  Train Loss: 0.3843
  Val Acc: 71.40%
  Val Precision: 0.7500
  Val Recall:    0.6420
  Val F1 Score:  0.6918
  Val Loss: 0.6927


# **10. Final test evaluation**

In [12]:
test_acc, test_prec, test_rec, test_f1, test_loss = evaluate(test_loader)

print("\n========== FINAL TEST ==========")
print(f"Test Acc:       {test_acc:.2f}%")
print(f"Test Precision: {test_prec:.4f}")
print(f"Test Recall:    {test_rec:.4f}")
print(f"Test F1 Score:  {test_f1:.4f}")
print(f"Test Loss:      {test_loss:.4f}")


========== FINAL TEST ==========
Test Acc:       69.90%
Test Precision: 0.7287
Test Recall:    0.6340
Test F1 Score:  0.6781
Test Loss:      0.6483
